# 04 · t-SNE y UMAP: ver en dos dimensiones sin engañarse

**Módulo 5 · Sesión 12** — Aprendizaje no supervisado

## Objetivos

PCA proyecta sobre un plano: conserva distancias grandes y aplasta lo demás. Para
**visualizar** estructura fina —qué puntos son vecinos de cuáles— hay métodos que
renuncian a las distancias globales y se concentran en los vecindarios: **t-SNE**
(van der Maaten y Hinton, 2008) y 🔵 **UMAP** (McInnes et al., 2018). Producen mapas
espectaculares y, por lo mismo, se malinterpretan a diario. Este notebook mide qué
conservan y qué inventan:

1. Comparar PCA, t-SNE y UMAP sobre los **dígitos manuscritos** (64 dimensiones), con
   tres medidas: cuántos vecinos se conservan (*trustworthiness*), qué tan bien clasifica
   un KNN en el mapa, y la silueta de las clases en el mapa.
2. Entender la idea de t-SNE —similitudes gaussianas arriba, t de Student abajo— y qué
   hacen la **perplejidad** y la semilla.
3. Medir sobre datos construidos lo que t-SNE **no conserva**: tamaños de grupos,
   distancias entre grupos, y que **inventa grupos** donde no los hay.
4. Ver qué añade UMAP: velocidad con muchos datos y `transform` para puntos nuevos.
5. Volver a **Wine Quality**: qué se ve en el mapa (tipo, dulce/seco) y qué no (calidad),
   y por qué un mapa 2D no es un conjunto de variables para un modelo.

La teoría está en `03-reduccion-dimensionalidad.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `umap-learn`.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")   # umap-learn y numba emiten avisos de versión
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Dígitos: tres mapas, tres medidas

1797 imágenes de 8 × 8 píxeles = 64 variables, 10 clases. Las etiquetas no las ve ningún
método; se usan solo para colorear y medir. Tres medidas de calidad de un mapa 2D:

- ***Trustworthiness*** (Venna y Kaski, 2001): fracción de los $k$ vecinos más cercanos
  de cada punto **en el mapa** que también eran vecinos en el espacio original (1 =
  ningún vecino falso). Mide lo que estos métodos prometen: vecindarios.
- **Accuracy de un KNN** (5 vecinos, CV) usando solo las dos coordenadas del mapa: si los
  vecinos en el mapa son de la misma clase, la estructura de clases sobrevivió.
- **Silueta de las clases reales** en el mapa: qué tan separadas se ven.

In [ ]:
digitos = load_digits()
X_dig, y_dig = digitos.data, digitos.target


def medir(X_alto, Z, y):
    return {"trustworthiness": trustworthiness(X_alto, Z, n_neighbors=5),
            "accuracy KNN (5) en el mapa": cross_val_score(KNeighborsClassifier(5), Z, y, cv=5).mean(),
            "silueta de las clases": silhouette_score(Z, y)}


mapas, tiempos = {}, {}
for nombre, metodo in [("PCA", PCA(n_components=2, random_state=SEMILLA)),
                       ("t-SNE", TSNE(n_components=2, perplexity=30, init="pca", random_state=SEMILLA)),
                       ("UMAP", umap.UMAP(n_components=2, random_state=SEMILLA))]:
    inicio = time.perf_counter()
    mapas[nombre] = metodo.fit_transform(X_dig)
    tiempos[nombre] = time.perf_counter() - inicio

tabla = pd.DataFrame({nombre: medir(X_dig, Z, y_dig) for nombre, Z in mapas.items()}).T
tabla["segundos"] = pd.Series(tiempos)
tabla.loc["(64 dimensiones originales)"] = [np.nan, cross_val_score(KNeighborsClassifier(5), X_dig, y_dig, cv=5).mean(),
                                             silhouette_score(X_dig, y_dig), np.nan]
print(tabla.round(3).to_string())

fig, ejes = plt.subplots(1, 3, figsize=(18, 5.5))
for eje, (nombre, Z) in zip(ejes, mapas.items()):
    disp = eje.scatter(Z[:, 0], Z[:, 1], c=y_dig, cmap="tab10", s=6)
    eje.set_title(f"{nombre} · KNN {tabla.loc[nombre, 'accuracy KNN (5) en el mapa']:.3f}")
    eje.set_xticks([])
    eje.set_yticks([])
plt.colorbar(disp, ax=ejes[2], ticks=range(10), label="dígito")
plt.show()

PCA conserva la geometría global —y con dos componentes de 64 (el 28 % de la varianza)
las clases se superponen: KNN 0.60, *trustworthiness* 0.83—. t-SNE y UMAP conservan los
**vecindarios**: *trustworthiness* 0.99, y un KNN sobre dos coordenadas clasifica
**mejor** que sobre las 64 originales (0.97 frente a 0.96). No es magia: el mapa se
construyó mirando todos los puntos, incluidos los que el KNN evalúa, y el método ya hizo
el trabajo de encontrar vecinos. Lo que sí dice es que la estructura de clases de los
dígitos es de **vecindarios** —cada dígito se parece a otros del mismo dígito—, y que
esa estructura vive en una variedad de baja dimensión dentro de las 64.

## 2. Qué hace t-SNE

Dos pasos. Arriba, en $p$ dimensiones, define para cada par de puntos una **similitud**
$p_{ij}$ con un núcleo gaussiano centrado en $\mathbf{x}_i$, cuya anchura $\sigma_i$ se
ajusta por punto para que el número efectivo de vecinos sea la **perplejidad**. Abajo, en
2D, define $q_{ij}$ con un núcleo **t de Student** de 1 grado de libertad (colas pesadas).
Después mueve los puntos del mapa por descenso del gradiente para minimizar la divergencia
de Kullback-Leibler $\text{KL}(P \,\|\, Q)$: que los vecinos arriba queden juntos abajo.

La t de Student es la idea clave. Con una gaussiana abajo, todos los puntos "medianamente
lejanos" se apretarían en el centro del mapa (el problema de *crowding*: en 2D no cabe
la misma cantidad de vecinos a distancia media que en 64D). Las colas pesadas permiten
que los puntos no vecinos se alejen mucho sin pagar casi costo:

In [ ]:
d = np.linspace(0, 5, 300)
gauss = np.exp(-d**2 / 2)
student = 1 / (1 + d**2)
fig, eje = plt.subplots(figsize=(7, 4))
eje.plot(d, gauss, label="gaussiana (arriba, en $p$ dimensiones)")
eje.plot(d, student, label="t de Student, 1 g.l. (abajo, en 2D)")
eje.set_xlabel("distancia")
eje.set_ylabel("similitud (sin normalizar)")
eje.set_title("Colas pesadas abajo: separar lo no vecino es barato")
eje.legend()
plt.show()
print(f"A distancia 3: gaussiana {np.exp(-9 / 2):.4f}, t de Student {1 / 10:.4f} (9 veces más)")

### Perplejidad y semilla

La perplejidad (5–50 es el rango habitual) decide cuántos vecinos "cuentan": pequeña,
el mapa se fragmenta en grupitos locales; grande, se parece más a la estructura global.
Y como la optimización es no convexa, el punto de partida importa: `init="pca"` (el valor
por defecto de `scikit-learn`, que usamos arriba) arranca de la proyección PCA y hace el
resultado reproducible; `init="random"` arranca al azar y cada semilla da un mapa
distinto — **rotado, reflejado, con los grupos en otro sitio**— con los mismos
vecindarios.

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(20, 4.8))
configs = [("perplejidad 5", dict(perplexity=5, init="pca", random_state=SEMILLA)),
           ("perplejidad 30", dict(perplexity=30, init="pca", random_state=SEMILLA)),
           ("perplejidad 100", dict(perplexity=100, init="pca", random_state=SEMILLA)),
           ("perplejidad 30, inicio aleatorio", dict(perplexity=30, init="random", random_state=SEMILLA + 1))]
filas = []
for eje, (nombre, cfg) in zip(ejes, configs):
    Z = TSNE(n_components=2, **cfg).fit_transform(X_dig)
    m = medir(X_dig, Z, y_dig)
    filas.append({"configuración": nombre, **m})
    eje.scatter(Z[:, 0], Z[:, 1], c=y_dig, cmap="tab10", s=5)
    eje.set_title(f"{nombre} · KNN {m['accuracy KNN (5) en el mapa']:.3f}")
    eje.set_xticks([])
    eje.set_yticks([])
plt.show()
print(pd.DataFrame(filas).set_index("configuración").round(3).to_string())

Los cuatro mapas conservan los vecindarios igual de bien (KNN 0.96–0.98). Lo que cambia
es la **apariencia**: con perplejidad 5 los dígitos se rompen en subgrupos; con 100, los
grupos son más redondos y compactos. Y el inicio aleatorio da otra disposición de los
mismos grupos. Ninguna de esas diferencias significa nada sobre los datos.

## 3. Lo que t-SNE no conserva, medido

El artículo *How to use t-SNE effectively* (Wattenberg et al., 2016) lo demuestra con
ejemplos interactivos; aquí lo medimos. Tres grupos gaussianos en 10 dimensiones: dos
compactos (desviación 1) y uno **cinco veces más disperso**; el tercero, además, **tres
veces más lejos** que los otros dos entre sí.

In [ ]:
def blobs_10d(n=250, dims=10):
    centros = np.zeros((3, dims))
    centros[1, 0] = 10           # grupo 1 a distancia 10 del 0
    centros[2, 0] = -30          # grupo 2 a distancia 30 del 0 (y 40 del 1)
    escalas = [1.0, 1.0, 5.0]
    X = np.vstack([centros[g] + rng.normal(0, escalas[g], (n, dims)) for g in range(3)])
    return X, np.repeat([0, 1, 2], n)


X_syn, y_syn = blobs_10d()
Z_pca = PCA(n_components=2).fit_transform(X_syn)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEMILLA).fit_transform(X_syn)


def dispersion(Z, y):
    """Desviación típica media de cada grupo respecto a su centroide."""
    return np.array([np.sqrt(((Z[y == g] - Z[y == g].mean(axis=0)) ** 2).sum(axis=1).mean()) for g in np.unique(y)])


def distancias_centroides(Z, y):
    c = np.array([Z[y == g].mean(axis=0) for g in np.unique(y)])
    return np.array([np.linalg.norm(c[0] - c[1]), np.linalg.norm(c[0] - c[2]), np.linalg.norm(c[1] - c[2])])


for nombre, Z in [("original (10D)", X_syn), ("PCA", Z_pca), ("t-SNE", Z_tsne)]:
    disp = dispersion(Z, y_syn)
    dist = distancias_centroides(Z, y_syn)
    print(f"{nombre:<15} dispersión grupo 2 / grupo 0 = {disp[2] / disp[0]:5.2f}   "
          f"distancia (0,2) / (0,1) = {dist[1] / dist[0]:5.2f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
for eje, (nombre, Z) in zip(ejes, [("PCA", Z_pca), ("t-SNE", Z_tsne)]):
    eje.scatter(Z[:, 0], Z[:, 1], c=y_syn, cmap="tab10", s=6, vmin=0, vmax=9)
    eje.set_title(nombre)
    eje.set_xticks([])
    eje.set_yticks([])
plt.show()

En los datos, el grupo 2 es 5 veces más disperso y está 3 veces más lejos. PCA lo conserva
(es una proyección lineal). En el mapa de t-SNE, los tres grupos tienen **casi el mismo
tamaño** y están a **distancias parecidas**: la perplejidad fija cuántos vecinos cuentan,
así que un grupo disperso se comprime hasta tener la misma densidad que uno compacto, y
la distancia entre grupos, que no es un vecindario de nadie, no se optimiza. **El tamaño
de un grupo en un mapa t-SNE no dice nada, y la distancia entre dos grupos, tampoco.**

### Grupos inventados

Peor: sobre una **sola** nube gaussiana en 10 dimensiones —sin ningún grupo—, t-SNE con
perplejidad baja dibuja grumos. Y la silueta de una partición K-Means sobre ese mapa es
varias veces mayor que sobre los datos, así que "clusterizar el mapa" encuentra
estructura que no existe.

In [ ]:
X_una = rng.normal(0, 1, (500, 10))
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
for eje, perp in zip(ejes, [5, 30, 100]):
    Z = TSNE(n_components=2, perplexity=perp, init="pca", random_state=SEMILLA).fit_transform(X_una)
    etiq = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit(Z).labels_
    eje.scatter(Z[:, 0], Z[:, 1], c=etiq, cmap="tab10", s=6, vmin=0, vmax=9)
    eje.set_title(f"Una gaussiana en 10D · perplejidad {perp}\nsilueta de K-Means (k=4) sobre el mapa: {silhouette_score(Z, etiq):.2f}")
    eje.set_xticks([])
    eje.set_yticks([])
plt.show()
etiq_orig = KMeans(n_clusters=4, n_init=10, random_state=SEMILLA).fit(X_una).labels_
print(f"Silueta de K-Means (k=4) sobre los datos originales en 10D: {silhouette_score(X_una, etiq_orig):.2f}")

En 10D, K-Means sobre la nube gaussiana da silueta 0.08 (no hay nada). Sobre el mapa de
t-SNE, la misma K-Means da 0.35–0.38 con cualquier perplejidad: el mapa **fabricó** la
apariencia de estructura "débil pero presente" (la misma escala de Kaufman y Rousseeuw
en la que Wine Quality dio 0.27 con grupos reales). Y con perplejidad 5 el mapa se rompe
en decenas de grumos que no corresponden a nada. Regla: **nunca** concluir que hay grupos a partir de un mapa t-SNE sin verificarlo
en los datos originales (referencia nula, notebook 02), y con más razón nunca ejecutar
K-Means sobre el mapa como si fuera un conjunto de variables.

Sobre los dígitos, donde sí hay grupos, K-Means sobre el mapa de t-SNE **sí** los
recupera mejor que en 64D (ARI 0.89 frente a 0.67), porque el mapa ya separó lo que en
64D se solapaba. Funciona cuando la estructura es real y engaña cuando no; la única forma
de distinguir los dos casos es medir en el espacio original.

In [ ]:
for nombre, Z in [("64 dimensiones", X_dig), ("PCA 2D", mapas["PCA"]), ("t-SNE 2D", mapas["t-SNE"]), ("UMAP 2D", mapas["UMAP"])]:
    etiq = KMeans(n_clusters=10, n_init=10, random_state=SEMILLA).fit(Z).labels_
    print(f"K-Means (k=10) sobre {nombre:<15} ARI contra el dígito real: {adjusted_rand_score(y_dig, etiq):.3f}")

## 4. 🔵 UMAP: parecido a t-SNE, con dos ventajas prácticas

UMAP construye un grafo de vecinos arriba y optimiza un grafo abajo con una función de
costo de entropía cruzada; en la práctica produce mapas parecidos a los de t-SNE,
conserva algo más de estructura global (los grupos que están lejos arriba tienden a quedar
lejos abajo), y tiene dos diferencias que importan:

- **Escala mejor**: t-SNE es $O(n \log n)$ con Barnes-Hut, pero su constante es alta;
  UMAP tarda más con pocos datos (la compilación de `numba`) y menos con muchos.
- Tiene **`transform`**: t-SNE solo puede mapear los puntos con los que se ajustó; UMAP
  puede colocar puntos nuevos en un mapa existente — imprescindible si el mapa se va a
  usar dentro de un `Pipeline`.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_dig, y_dig, test_size=0.3, stratify=y_dig, random_state=SEMILLA)
reductor = umap.UMAP(n_components=2, random_state=SEMILLA).fit(X_tr)
Z_tr, Z_te = reductor.transform(X_tr), reductor.transform(X_te)
knn = KNeighborsClassifier(5).fit(Z_tr, y_tr)
print(f"KNN entrenado en el mapa UMAP de entrenamiento, evaluado sobre puntos de prueba "
      f"colocados con transform: accuracy = {knn.score(Z_te, y_te):.3f}")
print("t-SNE de scikit-learn no tiene transform:", hasattr(TSNE(), "transform"))

fig, eje = plt.subplots(figsize=(6, 5.5))
eje.scatter(Z_tr[:, 0], Z_tr[:, 1], c=y_tr, cmap="tab10", s=5, alpha=0.3, label="entrenamiento")
eje.scatter(Z_te[:, 0], Z_te[:, 1], c=y_te, cmap="tab10", s=14, marker="x", label="prueba (transform)")
eje.set_title("UMAP: los puntos nuevos caen en el grupo de su dígito")
eje.set_xticks([])
eje.set_yticks([])
eje.legend()
plt.show()

## 5. Wine Quality en el mapa

Las 11 variables estandarizadas del notebook 02, con los tres métodos. Tres colores: el
tipo, el subgrupo dulce/seco de los blancos (K-Means del notebook 02), y la calidad.

In [ ]:
vinos = pd.read_csv("../datos/wine-quality.csv").drop_duplicates().reset_index(drop=True)
X = vinos.drop(columns=["quality", "tipo"])
X_esc = StandardScaler().fit_transform(X)
etiq_tipo = (vinos["tipo"] == "tinto").astype(int).to_numpy()
calidad = vinos["quality"].to_numpy()
buena = (calidad >= 7).astype(int)

mascara_b = etiq_tipo == 0
km_b = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(StandardScaler().fit_transform(X[mascara_b]))
dulce = int(np.argmax(X[mascara_b].groupby(km_b.labels_)["residual_sugar"].mean()))
subgrupo = np.full(len(vinos), 2)                                 # 2 = tinto
subgrupo[mascara_b] = (km_b.labels_ == dulce).astype(int)         # 1 = blanco dulce, 0 = blanco seco

mapas_vino, tiempos_vino = {}, {}
for nombre, metodo in [("PCA", PCA(n_components=2, random_state=SEMILLA)),
                       ("t-SNE", TSNE(n_components=2, perplexity=30, init="pca", random_state=SEMILLA)),
                       ("UMAP", umap.UMAP(n_components=2, random_state=SEMILLA))]:
    inicio = time.perf_counter()
    mapas_vino[nombre] = metodo.fit_transform(X_esc)
    tiempos_vino[nombre] = time.perf_counter() - inicio

fig, ejes = plt.subplots(3, 3, figsize=(16, 14))
for col, (nombre, Z) in enumerate(mapas_vino.items()):
    for fila, (color, cmap, titulo) in enumerate([(etiq_tipo, "viridis", "tipo"),
                                                  (subgrupo, "tab10", "tinto / blanco seco / blanco dulce"),
                                                  (calidad, "viridis", "quality")]):
        eje = ejes[fila, col]
        eje.scatter(Z[:, 0], Z[:, 1], c=color, cmap=cmap, s=3, alpha=0.6, vmin=0 if cmap == "tab10" else None,
                    vmax=9 if cmap == "tab10" else None)
        eje.set_title(f"{nombre} · color: {titulo}" + (f" · {tiempos_vino[nombre]:.1f} s" if fila == 0 else ""))
        eje.set_xticks([])
        eje.set_yticks([])
plt.tight_layout()
plt.show()

Los tres mapas muestran lo mismo que K-Means y PCA ya habían dicho: tinto/blanco es la
estructura dominante, y dentro de los blancos hay un gradiente dulce/seco (más nítido en
t-SNE y UMAP, que lo separan en dos lóbulos). Y en la tercera fila, la calidad: **no se
ve**. Los vinos buenos están repartidos por todo el mapa, con una leve concentración en
la zona de los blancos secos. Se mide igual que en los dígitos:

In [ ]:
filas = []
for nombre, Z in mapas_vino.items():
    filas.append({"mapa": nombre,
                  "trustworthiness": trustworthiness(X_esc, Z, n_neighbors=5),
                  "KNN: accuracy tipo": cross_val_score(KNeighborsClassifier(5), Z, etiq_tipo, cv=5).mean(),
                  "KNN: AP quality ≥ 7": cross_val_score(KNeighborsClassifier(5), Z, buena, cv=5, scoring="average_precision").mean()})
filas.append({"mapa": "11 dimensiones originales", "trustworthiness": np.nan,
              "KNN: accuracy tipo": cross_val_score(KNeighborsClassifier(5), X_esc, etiq_tipo, cv=5).mean(),
              "KNN: AP quality ≥ 7": cross_val_score(KNeighborsClassifier(5), X_esc, buena, cv=5, scoring="average_precision").mean()})
print(pd.DataFrame(filas).set_index("mapa").round(3).to_string())

El tipo se conserva en cualquier mapa (accuracy 0.98–0.99). La calidad, no: la AP de un
KNN cae de 0.38 en 11D a 0.28–0.36 en 2D. Es el mismo mensaje del notebook 03 (sección
5): la información que separa los vinos buenos está repartida en direcciones de poca
varianza y en vecindarios finos que dos coordenadas no pueden guardar. Un mapa 2D sirve
para **mirar**; para predecir, se usan las 11 variables (o, como en el módulo 4, un
ensamble sobre ellas).

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué conserva cada método? | PCA: distancias globales (dígitos: KNN 0.60 en 2D). t-SNE y UMAP: vecindarios (*trustworthiness* 0.99, KNN 0.97) |
| ¿Perplejidad e inicialización? | Cambian la apariencia (fragmentación, disposición), no los vecindarios (KNN 0.96–0.98 en todos) |
| ¿Qué no conserva t-SNE? | Tamaño de los grupos (5× en los datos → ≈1× en el mapa), distancias entre grupos (3× → ≈1×) |
| ¿Inventa grupos? | Sí: una gaussiana en 10D da un mapa con grumos y silueta 0.35–0.38 para K-Means (0.08 en los datos) |
| ¿Clusterizar el mapa? | Funciona si hay grupos reales (dígitos: ARI 0.89 frente a 0.67 en 64D) y engaña si no; verificar siempre en el espacio original |
| ¿UMAP? | Mapas parecidos, más estructura global, `transform` para puntos nuevos (KNN 0.98 sobre prueba), escala mejor |
| ¿Wine? | Tipo y dulce/seco se ven; la calidad no, y un KNN pierde AP al pasar de 11D (0.38) a 2D (0.28–0.36) |